# Word embeddings

This tutorial contains an introduction to word embeddings. You will train your own word embeddings using a simple pytorch model for a sentiment classification task, and then visualize.

<img src="https://github.com/tensorflow/text/blob/master/docs/tutorials/images/embedding.jpg?raw=1" alt="Screenshot of the embedding projector" width="400"/>



## Setup

In [ ]:

!pip install "numpy<2"
!pip install torch==2.0.1+cu117 -f https://download.pytorch.org/whl/torch_stable.html
!pip install torchtext==0.15.2



In [ ]:
import os
import re
import string
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchtext.datasets import IMDB
from torchtext.data.utils import get_tokenizer
from collections import Counter
from torchtext.vocab import Vocab
import numpy as np
import io
import os
import tarfile
import urllib.request
import shutil
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import pandas as pd
import numpy as np

### Download the IMDb Dataset
You will use the [Large Movie Review Dataset](http://ai.stanford.edu/~amaas/data/sentiment/) through the tutorial. You will train a sentiment classifier model on this dataset and in the process learn embeddings from scratch. To read more about loading a dataset from scratch, see the [Loading text tutorial](https://www.tensorflow.org/tutorials/load_data/text).  



In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
filename = "aclImdb_v1.tar.gz"
extract_dir = "aclImdb"

if not os.path.exists(filename):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)

if not os.path.exists(extract_dir):
    print("Extracting dataset...")
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall()
print("Dataset ready.")

Extracting dataset...
Dataset ready.



# Text Preprocessing Function Class
  

- Sets up stopword removal and lemmatization.
- Defines clean_text() to:
    - Lowercase text
    - Remove HTML breaks and punctuation
    - Remove stopwords and lemmatize each word
    - Return the cleaned and normalized text

# IMDbDataset Class
*  Loads and preprocesses all reviews (positive and negative) for a given split (train/test).

*  Cleans each review using clean_text().
    
*  Builds a vocabulary from the most frequent words if not provided.

*  Converts each review to a list of word indices, pads/truncates to a fixed length, and returns the tensor and label.

    


In [ ]:
def clean_text(text):
    # remove HTML breaks
    text = text.replace('<br />', ' ')
    #lower text
    text = text.lower()
    # remove punctuation
    for punctation in [',;.!?;']:
        text = text.replace(punctation, '')
    #to be completed
    return text

class IMDbDataset(Dataset):
    def __init__(self, root_dir, split, vocab=None, max_len=100):
        self.texts = []
        self.labels = []
        self.vocab = vocab
        self.max_len = max_len

        # Load positive and negative reviews
        for label, sentiment in [(1, "pos"), (0, "neg")]:
            path = os.path.join(root_dir, split, sentiment)
            for file in os.listdir(path):
                with open(os.path.join(path, file), 'r', encoding='utf-8') as f:
                    text = f.read()
                    text = clean_text(text)
                    text = re.sub('<br />', ' ', text.lower())
                    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
                    self.texts.append(text)
                    self.labels.append(label)

        # Build vocabulary if not provided
        if vocab is None:
            counter = Counter()
            for text in self.texts:
                counter.update(text.split())
            self.vocab = {"<pad>": 0, "<unk>": 1}
            for i, (word, _) in enumerate(counter.most_common(9998)):
                self.vocab[word] = i + 2

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx].split()
        # Convert words to indices
        indices = [self.vocab.get(word, self.vocab["<unk>"]) for word in text[:self.max_len]]
        # Pad sequence
        if len(indices) < self.max_len:
            indices += [self.vocab["<pad>"]] * (self.max_len - len(indices))

        return torch.tensor(indices), torch.tensor(self.labels[idx], dtype=torch.float32)


# Create Datasets and DataLoaders

In [ ]:
# Create datasets 
root_dir = "./aclImdb"  # chemin du dossier IMDb après téléchargement/décompression
train_dataset = IMDbDataset(root_dir=root_dir, split="train")
test_dataset = IMDbDataset(root_dir=root_dir, split="test", vocab=train_dataset.vocab)  # utiliser le même vocabulaire

# Create validation set (80/20 split)
from torch.utils.data import random_split
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Create data loaders 
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# prompt: take a look at the dataset

print(len(train_dataset))
train_dataset[0]


20000


(tensor([   1,   36,    1,   42,   86,   11,   17,   13,  600,   41,   12,    9,
           67,   57,  112,  219,   25,   74,  282,   15, 1852,  849,    1, 4886,
           11,   19,   13,   90,    6,   26,    4, 1488,  498,    5,  217,    3,
            8,   57,   94, 2199,    4,  112,    8,   99,   94, 3109,   41,  794,
           10,   58,   77,   38,    6, 2936,  129,    1,   12,   44,   22,   23,
          165,    6,    1,    2,  600,   45,    5,   11,   19,   12,   34,  293,
            6,   26,  978,    2,  210,   11,   19,    7,  678,   90,   15,   81,
           36,   89, 1110,    2,   37,  487,  314, 1238,   60,  180,    6,   69,
           40,    4, 2422,    5]),
 tensor(1.))

## Create a classification model

    - Defines a neural network:

    - Embedding layer to turn word indices into vectors

    - Global average pooling (mean over sequence)

    - Dense layer with 16 units and ReLU activation

    - Output layer for binary sentiment classification


In [ ]:
class CBOWSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, 16)
        self.relu = nn.ReLU()
        self.output = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch_size, sequence_length)
        embedded = self.embedding(x)  # (batch_size, sequence_length, embedding_dim)
        pooled = embedded.mean(dim=1)  # global average pooling: (batch_size, embedding_dim)
        hidden = self.relu(self.fc1(pooled))  # (batch_size, 16)
        out = self.sigmoid(self.output(hidden))  # (batch_size, 1)
        return out

## Compile and train the model

**

Compile and train the model using the Adam optimizer and BinaryCrossentropy loss.
**

In [ ]:
# Instantiate model
vocab_size = len(train_dataset.dataset.vocab)  # car train_dataset est un Subset
model = CBOWSentimentClassifier(vocab_size)

# Optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



Trains the model for a number of epochs:

- For each batch, computes predictions, loss, gradients, and updates model weights.

- Calculates training accuracy.

- Evaluates the model on the validation set and prints accuracy and loss for each epoch.



In [ ]:
# Training loop

epochs = 400
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.unsqueeze(1)  # (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(inputs)  # (batch_size, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        preds = (outputs >= 0.5).float()  # threshold à 0.5
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.unsqueeze(1)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            preds = (outputs >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch+1}/{epochs}, "
          f"Loss: {total_loss/total:.4f}, Train Acc: {correct/total:.4f}, "
          f"Val Loss: {val_loss/val_total:.4f}, Val Acc: {val_correct/val_total:.4f}")

Visualize the model metrics in TensorBoard.

In [ ]:
# Get embeddings and vocabulary
weights = model.embedding.weight.data.cpu().numpy()
vocab_list = list(train_dataset.dataset.vocab.keys())

# Save embeddings and vocabulary
with io.open('vectors.tsv', 'w', encoding='utf-8') as out_v, \
     io.open('metadata.tsv', 'w', encoding='utf-8') as out_m:
    for i, word in enumerate(vocab_list):
        if i == 0:  # Skip padding token
            continue
        vec = weights[i]
        out_v.write('\t'.join([str(x) for x in vec]) + "\n")
        out_m.write(word + "\n")

print("Embeddings saved to vectors.tsv and metadata.tsv")


Embeddings saved to vectors.tsv and metadata.tsv


In [ ]:


# Load embeddings
embeddings = np.loadtxt('vectors.tsv', delimiter='\t')

# Load vocabulary (words)
with open('metadata.tsv', 'r', encoding='utf-8') as f:
    words = [line.strip() for line in f.readlines()]

print(f'Loaded {len(words)} words and embeddings of shape {embeddings.shape}')


Loaded 9999 words and embeddings of shape (9999, 16)


In [ ]:


# Option 1: PCA (fast, linear)
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)
df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'word': words
})
# Option 2: t-SNE (slower, nonlinear, often better for visualization)
# tsne = TSNE(n_components=2, random_state=42)
# embeddings_2d = tsne.fit_transform(embeddings)
import plotly.express as px

fig = px.scatter(df, x='x', y='y', hover_name='word',
                 title='Word Embeddings Visualization (2D PCA)',
                 width=800, height=600)

fig.show()


In [ ]:
# Highlight words containing 'were'
df['highlight'] = df['word'].apply(lambda w: 'were' in w.lower())

fig = px.scatter(df, x='x', y='y', hover_name='word', color='highlight',
                 title='Word Embeddings Visualization with Highlight',
                 width=800, height=600)

fig.show()
